<a href="https://colab.research.google.com/github/Innovatewithapple/TransformersProjects/blob/main/BERTCustomModelPytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader,Dataset

In [3]:
autotoken = AutoTokenizer.from_pretrained('bert-base-uncased')
autotoken.vocab_size

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

30522

In [4]:
raw_datasets = load_dataset('imdb')

train_Dataset = raw_datasets['train']
val_Dataset = raw_datasets['test']

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [30]:
embed_dim=256
vocab_size=autotoken.vocab_size
max_len=200
num_heads=8
ff_dim= 4 * embed_dim
num_layers=4
num_classes=2

In [31]:
class IMDBDataset(Dataset):
  def __init__(self,text,labels,max_len):
    self.text = text
    self.tokenizer = autotoken
    self.lables = labels
    self.max_len = max_len

  def __len__(self):
    return len(self.text)

  def __getitem__(self,idx):
    # clean the text, real world has messy or nan
    item = str(self.text[idx])

    encoding = self.tokenizer(text=item,padding='max_length',max_length=self.max_len,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')

    return {
        'input_ids':encoding['input_ids'].flatten(),
        'attention_mask': encoding['attention_mask'].flatten(),
        'labels': torch.tensor(self.lables[idx],dtype=torch.long)
    }


In [32]:
train_data = IMDBDataset(text=train_Dataset['text'],labels=train_Dataset['label'],max_len=max_len)
val_data = IMDBDataset(text=val_Dataset['text'],labels=val_Dataset['label'],max_len=max_len)

In [33]:
train_loader = DataLoader(dataset=train_data,batch_size=32,shuffle=True,pin_memory=True)
val_loader = DataLoader(dataset=val_data,batch_size=32,shuffle=False,pin_memory=True)

In [34]:
class TransformerBlock(nn.Module):
  def __init__(self,embed_dim,num_heads,ff_dim) -> None:
    super().__init__()
    self.attention = nn.MultiheadAttention(embed_dim=embed_dim,num_heads=num_heads,dropout=0.2)

    self.mlp = nn.Sequential(
        nn.Linear(in_features=embed_dim,out_features=ff_dim),
        nn.GELU(),
        nn.Linear(in_features=ff_dim,out_features=embed_dim),
        nn.Dropout(0.4)
    )

    self.layernorm1 = nn.LayerNorm(normalized_shape=embed_dim)
    self.layernorm2 = nn.LayerNorm(normalized_shape=embed_dim)

  def forward(self,inputtext):
    att,_ = self.attention(inputtext,inputtext,inputtext)
    merged_att = self.layernorm1(inputtext + att)
    mlp_output = self.mlp(merged_att)
    return self.layernorm2(merged_att + mlp_output)


In [35]:
class WordEmbedding(nn.Module):
  def __init__(self,embed_dim,vocab_size) -> None:
    super().__init__()

    self.wordEmbed = nn.Embedding(num_embeddings=vocab_size,embedding_dim=embed_dim)

  def forward(self,x):
    return self.wordEmbed(x)

In [36]:
class NLPTransformer(nn.Module):
  def __init__(self,embed_dim,vocab_size,max_len,num_heads,ff_dim,num_layers,num_classes) -> None:
    super().__init__()
    self.wordEmbed = WordEmbedding(embed_dim=embed_dim,vocab_size=vocab_size)
    self.positionEmbed = nn.Parameter(torch.zeros(1,max_len,embed_dim))

    self.transform_layers = nn.ModuleList([
        TransformerBlock(embed_dim=embed_dim,num_heads=num_heads,ff_dim=ff_dim)
        for _ in range(num_layers)
    ])
    self.dropout = nn.Dropout(0.4)
    self.output_layer = nn.Linear(in_features=embed_dim,out_features=num_classes)

  def forward(self,x,mask):
    x = self.wordEmbed(x) + self.positionEmbed
    x = self.dropout(x)

    for transformer_layer in self.transform_layers:
      x = transformer_layer(x)

    # 3. THE SMART MEAN (Replacing x = x.mean(dim=1))
    # We need to make the mask (Batch, 512) match x (Batch, 512, 128)
    mask = mask.unsqueeze(-1) # shape becomes [Batch, 512, 1]
    x = x * mask
    # Sum only the real words and divide by the count of real words
    x = x.sum(dim=1) / mask.sum(dim=1)
    return self.output_layer(x)

In [37]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [38]:
model = NLPTransformer(embed_dim=embed_dim,
                       vocab_size=vocab_size,
                       max_len=max_len,
                       num_heads=num_heads,
                       ff_dim=ff_dim,
                       num_layers=num_layers,
                       num_classes=num_classes).to(device)

In [39]:
optimizer = optim.Adam(params=model.parameters(),lr=1e-5,weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
epochs = 12

for epoch in range(epochs):
  model.train()
  train_loss = 0
  train_correct = 0
  train_total = 0

  for batch in train_loader:
     # 1. Unpack all three items from your Dictionary
    input_ids = batch['input_ids'].to(device)
    mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    optimizer.zero_grad()

    output = model(input_ids,mask)
    loss = loss_fn(output,labels)

    loss.backward()
    optimizer.step()

    train_loss += loss.item()
    pred = torch.argmax(input=output,dim=1)
    train_correct += (pred == labels).sum().item()
    train_total += labels.size(0)

  train_accuracy = train_correct / train_total
  train_losses = train_loss / len(train_loader)

  model.eval()
  val_loss = 0
  val_correct = 0
  val_total = 0

  with torch.no_grad():
    for batch in val_loader:
     # 1. Unpack all three items from your Dictionary
     input_ids = batch['input_ids'].to(device)
     mask = batch['attention_mask'].to(device)
     labels = batch['labels'].to(device)

     output = model(input_ids,mask)
     loss = loss_fn(output,labels)

     val_loss += loss.item()
     pred = torch.argmax(input=output,dim=1)
     val_correct += (pred == labels).sum().item()
     val_total += labels.size(0)

    val_accuracy = val_correct / val_total
    val_losses = val_loss / len(val_loader)

    print(f'\nEpochs: {epoch+1}/{epochs}...')
    print(f'Train_acc: {train_accuracy} | Train_loss: {train_losses}')
    print(f'Val_acc: {val_accuracy} | Val_loss: {val_losses}')
    print('==============================================================')


Epochs: 1/12...
Train_acc: 0.57316 | Train_loss: 0.6801804624250173
Val_acc: 0.56724 | Val_loss: 0.6800444389853026

Epochs: 2/12...
Train_acc: 0.63156 | Train_loss: 0.6525152385082391
Val_acc: 0.613 | Val_loss: 0.6536361607520477

Epochs: 3/12...
Train_acc: 0.6592 | Train_loss: 0.6251606640532194
Val_acc: 0.60912 | Val_loss: 0.661182510387867

Epochs: 4/12...
Train_acc: 0.67872 | Train_loss: 0.6010071397818568
Val_acc: 0.61756 | Val_loss: 0.6610784717380543

Epochs: 5/12...
Train_acc: 0.68676 | Train_loss: 0.5908870229025935
Val_acc: 0.61808 | Val_loss: 0.6668670409933075

Epochs: 6/12...
Train_acc: 0.69292 | Train_loss: 0.5823821329399753
Val_acc: 0.624 | Val_loss: 0.6723802568357619

Epochs: 7/12...
Train_acc: 0.70024 | Train_loss: 0.5767722081421586
Val_acc: 0.62388 | Val_loss: 0.669343834642864

Epochs: 8/12...
Train_acc: 0.7072 | Train_loss: 0.5681310529294221
Val_acc: 0.63284 | Val_loss: 0.6598193276949855

Epochs: 9/12...
Train_acc: 0.70804 | Train_loss: 0.5647521459919107
Val